# A local Oracle AI Database for System One experiments

An experiment needs durable evidence: source text, vector embeddings, labels, decisions,
and measured usage. Here we prepare **Oracle AI Database in Docker**, connect using
the thin Python driver, and run a real vector-distance query. The five lessons then
build their own SQL and HTTP functions from scratch.

**Learning goals:** understand the container/database/PDB distinction, keep API keys
out of notebooks, create a restricted lab account, and verify native VECTOR support.
Run this notebook locally. A hosted notebook's `localhost` is a different computer.

The compose file publishes Oracle only on `127.0.0.1:1535`. `FREEPDB1` is the pluggable
database service; `s1lab` is the course's account inside it. The Docker volume preserves
data between container restarts. Stopping the container does not delete that volume.

## Part 1 · Install prerequisites

Install and start [Docker Desktop](https://docs.docker.com/desktop/), with enough memory
and disk for an Oracle Free container (allow several GB of RAM and at least 15 GB free
disk for image, database and model weights). Use Python 3.12 and a local Jupyter kernel.

The published image tag can change. The compose file uses Oracle's `latest-lite`; record
the actual image ID and database version for reproducibility. On Apple Silicon, the
default compose platform uses amd64 emulation. Use an appropriate verified native Oracle
image/platform if your deployment requires it. Docker image availability and initialization
time depend on the host. Do not assume an image's tag is the database's actual version.

In [ ]:
%pip install -q "oracledb>=4.0,<5" "python-dotenv>=1,<2"

In [1]:
import array
import pprint
import os
import re
import secrets
import subprocess
from getpass import getpass
from pathlib import Path
import oracledb
from dotenv import load_dotenv

load_dotenv('.env')

True

## Part 2 · Configure the lab

Copy `.env.example` to `.env` for the database connection settings. It is gitignored.
The experiment notebooks prompt for API keys using `getpass`; no API key is needed
for this database-only setup. Never paste keys into code cells or save them in outputs.
These cells prompt for missing database credentials. A fresh container uses a new administrator password; an existing container
keeps its existing password. The lessons use the restricted lab password, not SYSTEM.

**Fresh installation:** set `START_NEW_CONTAINER=True` and `CREATE_LAB_USER=True`.
**Existing Docker Oracle:** leave startup off, set `ORACLE_DSN` to its published port,
and create the lab user once if it does not exist. Existing data is never dropped.

In [2]:
START_NEW_CONTAINER = False
CREATE_LAB_USER = False
os.environ.setdefault('ORACLE_DSN', '127.0.0.1:1535/FREEPDB1')
os.environ.setdefault('ORACLE_USER', 's1lab')
if not os.getenv('ORACLE_PASSWORD'):
    os.environ['ORACLE_PASSWORD'] = getpass('Lab account password: ')
if START_NEW_CONTAINER or CREATE_LAB_USER:
    if not os.getenv('ORACLE_ADMIN_PASSWORD'):
        os.environ['ORACLE_ADMIN_PASSWORD'] = getpass('Oracle SYSTEM password: ')

## Part 3 · Start Oracle with Docker Compose

The command creates only this course's container and volume. Initialization can take
several minutes. Inspect `docker compose ps` until the health status is healthy. If the
host already has a suitable Oracle container, reuse it with an isolated account instead
of creating a second database. Never delete someone else's volumes to free space.

You can also run these commands in a terminal from this folder:

```bash
docker compose up -d
docker compose ps
python bootstrap_oracle.py
```

Use `docker compose stop` to stop your new course container later. Avoid `down --volumes`
unless you intentionally want to delete this course's database.

In [3]:
if START_NEW_CONTAINER:
    subprocess.run(['docker', 'compose', 'up', '-d'], check=True)
    print('Wait for healthy status before running the account-creation cell.')
else:
    print('Using the existing Oracle endpoint configured in the environment.')
subprocess.run(['docker', 'version', '--format', '{{.Server.Version}}'], check=True)

Using the existing Oracle endpoint configured in the environment.
28.5.1


CompletedProcess(args=['docker', 'version', '--format', '{{.Server.Version}}'], returncode=0)

In [4]:
if START_NEW_CONTAINER:
    subprocess.run(['docker', 'compose', 'ps'], check=True)
else:
    print('Check the health of your existing Oracle Docker container.')

Check the health of your existing Oracle Docker container.


## Part 4 · Create a restricted account once

DDL identifiers cannot be bound like ordinary values. Validate the account name and
password character set before constructing these statements. The account can connect,
create its own tables, and use up to 250 MB in USERS. It cannot administer other users.
If it already exists, preserve its password and data. This cell needs the existing
container's SYSTEM password; the experiment cells do not.

In [5]:
user = os.environ['ORACLE_USER']
password = os.environ['ORACLE_PASSWORD']
if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]{0,29}', user):
    raise ValueError('Choose a simple Oracle username.')
if not re.fullmatch(r'[A-Za-z0-9_!#-]{12,100}', password):
    raise ValueError('Use 12–100 letters, digits, underscores, !, # or -.')

In [6]:
if CREATE_LAB_USER:
    with oracledb.connect(user='system', password=os.environ['ORACLE_ADMIN_PASSWORD'],
                         dsn=os.environ['ORACLE_DSN']) as admin:
        with admin.cursor() as cur:
            cur.execute("""
                SELECT COUNT(*)
                FROM all_users
                WHERE username = :1
            """, [user.upper()])
            if cur.fetchone()[0] == 0:
                cur.execute(f'CREATE USER {user} IDENTIFIED BY "{password}"')
                cur.execute(f'GRANT CREATE SESSION, CREATE TABLE TO {user}')
                cur.execute(f'ALTER USER {user} QUOTA 250M ON USERS')
                print('Created the isolated lab account.')
            else:
                print('Account exists; credentials and data preserved.')

## Part 5 · Verify the database and VECTOR arithmetic

The Python driver connects in thin mode: no Oracle Client installation is needed.
The two example vectors below are orthogonal, so their cosine distance is 1. We use
tiny hand-written vectors here to test the SQL type. **These are not substituted for
Voyage embeddings in any experiment.** The next lessons call Voyage for real 1,024-dimensional
document/query vectors, store them in `VECTOR(1024, FLOAT32)`, and retrieve with SQL.

In [7]:
db = oracledb.connect(user=user, password=password, dsn=os.environ['ORACLE_DSN'])
with db.cursor() as cur:
    cur.execute("""
        SELECT
            SYS_CONTEXT('USERENV', 'CURRENT_USER') AS account_name,
            SYS_CONTEXT('USERENV', 'CON_NAME') AS container_name
        FROM dual
    """)
    account, container = cur.fetchone()
    pprint.pprint({'account': account, 'container': container}, sort_dicts=False)
    cur.execute("""
        SELECT banner_full
        FROM v$version
    """)
    print(cur.fetchone()[0])

{'account': 'S1LAB', 'container': 'FREEPDB1'}
Oracle AI Database 26ai Free Release 23.26.2.0.0 - Develop, Learn, and Run for Free
Version 23.26.2.0.0


In [8]:
with db.cursor() as cur:
    cur.execute("""
        SELECT VECTOR_DISTANCE(:a, :b, COSINE) AS distance
        FROM dual
    """,
                a=array.array('f', [1, 0, 0]), b=array.array('f', [0, 1, 0]))
    distance = float(cur.fetchone()[0])
assert abs(distance - 1.0) < 1e-6
pprint.pprint({'vector_check': 'passed', 'expected_distance': 1.0,
               'observed_distance': distance}, sort_dicts=False)
db.close()

{'vector_check': 'passed', 'expected_distance': 1.0, 'observed_distance': 1.0}


## Part 6 · Choose your first lesson

1. **01 — Reranking:** freeze Oracle candidates, compare seven ranking methods, then use a fixed OpenAI reader.
2. **02 — Summary quality:** verify supported claims and action-critical coverage before admitting a summary.
3. **03 — Memory routing:** distinguish deciding to search from deciding to inject already-retrieved evidence.
4. **04 — Tool/skill selection:** compare eligible-catalog matching and abstention; selection is not execution permission.
5. **05 — Chunking:** preserve every source character and evaluate retrieved source spans at equal context budgets.
6. **06 — Reader comparison:** discover real model IDs and compare readers on identical evidence.

Get keys for [TypeSafe](https://docs.typesafe.ai/introduction/quickstart),
[Voyage](https://docs.voyageai.com/docs/api-key-and-installation), and
[OpenAI](https://developers.openai.com/api/docs/quickstart). The Claude reader extension
also needs an Anthropic key. The notebooks ask for API keys using hidden `getpass`
prompts. The appbook reads its server-side keys from `.env`.

Each experiment creates its own run ID and records real costs and decisions in Oracle.
The appbook reads those same records. In SQL, `s1_documents` holds sources/vectors,
`s1_runs` holds configuration and the call ledger, and `s1_results` holds case/method rows.

**Check your understanding:** why should a reranker use one frozen candidate pool?
Why is a probability concentrated on one option different from proof that it is true?
Why does adding rows to Oracle not constitute updating model weights?

References: [Oracle Free container](https://container-registry.oracle.com/ords/ocr/ba/database/free),
[python-oracledb VECTOR support](https://python-oracledb.readthedocs.io/en/latest/user_guide/vector_data_type.html).